In [ ]:
# @title Setup
from google.cloud import bigquery
from google.colab import data_table
import bigframes.pandas as bpd

project = 'smiling-rhythm-486213-b9'
location = 'US'
client = bigquery.Client(project=project, location=location)
data_table.enable_dataframe_formatter()

## I/ Market Demand & Seasonality

### Question 1: How does the demand for yellow taxis fluctuate over time (daily, weekly, monthly, and seasonally)?


In [ ]:
query_demand_over_time = """
SELECT *
FROM `smiling-rhythm-486213-b9.views_fordashboard.demand_over_time`
"""

demand_over_time_df = client.query(query_demand_over_time).to_dataframe()
demand_over_time_df

In [ ]:
type(demand_over_time_df)

In [ ]:
demand_over_time_df.info()

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
demand_over_time_df['trip_date'] = pd.to_datetime(demand_over_time_df['trip_date'])
demand_over_time_df.info()

In [ ]:
demand_over_time_df.year.value_counts(normalize=True)

In [ ]:
from datetime import datetime
current_year = datetime.now().year
current_year

In [ ]:
filtered_demand_over_time_df = demand_over_time_df[(demand_over_time_df['year'] >= 2020) & (demand_over_time_df['year'] <= current_year)]
filtered_demand_over_time_df.year.value_counts(normalize=True)

In [ ]:
fig_daily = px.line(
    filtered_demand_over_time_df,
    x='trip_date',
    y='total_trips',
    title='Daily Taxi Demand Over Time',
    labels={'trip_date': 'Date', 'total_trips': 'Number of Trips'},
    template='plotly_dark'
)
fig_daily.show()

In [ ]:
weekly_demand = filtered_demand_over_time_df.groupby('week')['total_trips'].sum().reset_index()
fig_weekly = px.line(
    weekly_demand,
    x='week',
    y='total_trips',
    title='Weekly Taxi Demand Trend',
    labels={'week': 'Week Number', 'total_trips': 'Number of Trips'},
    template='plotly_dark'
)
fig_weekly.show()

In [ ]:
monthly_demand = filtered_demand_over_time_df.groupby('month')['total_trips'].sum().reset_index()
fig_monthly = px.bar(
    monthly_demand,
    x='month',
    y='total_trips',
    title='Monthly Taxi Demand Trend',
    labels={'month': 'Month', 'total_trips': 'Number of Trips'},
    template='plotly_dark'
)
fig_monthly.show()

In [ ]:
filtered_demand_over_time_df['quarter'] = filtered_demand_over_time_df['trip_date'].dt.to_period("Q")
filtered_demand_over_time_df

In [ ]:
seasonal_demand = filtered_demand_over_time_df.groupby('quarter')['total_trips'].sum().reset_index()
seasonal_demand

In [ ]:
seasonal_demand['quarter'] = seasonal_demand['quarter'].astype(str)

In [ ]:
fig_seasonal = px.line(
    seasonal_demand,
    x='quarter',
    y='total_trips',
    title='Seasonal Taxi Demand Trend',
    labels={'quarter': 'Quarter', 'total_trips': 'Number of Trips'},
    template='plotly_dark'
)
fig_seasonal.show()

### Question 2: What are the peak hours for yellow taxi trips in different boroughs and zones?


In [ ]:
def query_to_dataframe(query: str) -> pd.DataFrame:
    """
    Executes a SQL query in BigQuery and returns a Pandas DataFrame.
    """
    try:
        df = client.query(query).to_dataframe()
        print(f"Query executed successfully. Retrieved {df.shape[0]} rows.")
        return df
    except Exception as e:
        print(f"Error executing query: {e}")
        return pd.DataFrame()

In [ ]:
query_peak_hours_by_zone = """
SELECT *
FROM `smiling-rhythm-486213-b9.views_fordashboard.peak_hours_by_zone`
"""
peak_hours_by_zone_df = query_to_dataframe(query_peak_hours_by_zone)
peak_hours_by_zone_df.head()

In [ ]:
peak_hours_by_zone_df.info()

In [ ]:
fig_borough = px.line(
    peak_hours_by_zone_df.groupby(["pickup_hour", "Borough"])["total_trips"].sum().reset_index(),
    x="pickup_hour",
    y="total_trips",
    color="Borough",
    title="Viewing peak hours by Borough",
    labels={"pickup_hour": "Heure", "total_trips": "Nombre de courses"},
    template="plotly_dark"
)

fig_borough.show()

In [ ]:
pivot_df = peak_hours_by_zone_df.pivot_table(
    values="total_trips",
    index="pickup_hour",
    columns="Borough",
    aggfunc="sum"
)

pivot_df.head()

In [ ]:
fig_zone = go.Figure(
    data=go.Heatmap(
        z=pivot_df.values,
        x=pivot_df.columns,
        y=pivot_df.index,
        colorscale="YlGnBu"
    )
)

fig_zone.update_layout(
    title="Heatmap des Heures de Pointe par Arrondissement",
    xaxis_title="Borough",
    yaxis_title="Hour",
    template="plotly_dark"
)

fig_zone.show()

In [ ]:
borough_hourly_demand = peak_hours_by_zone_df.groupby(["Borough", "pickup_hour"])["total_trips"].sum().reset_index()
borough_hourly_demand.head()

In [ ]:
fig_facet = px.bar(
    borough_hourly_demand,
    x="pickup_hour",
    y="total_trips",
    color="Borough",
    facet_col="Borough",
    facet_col_wrap=3,
    title="Peak Hours for Yellow Taxi Trips by Borough",
    labels={"pickup_hour": "Hour of the Day", "total_trips": "Total Trips"},
    template="plotly_dark"
)

fig_facet.update_layout(
    showlegend=False,
    height=600,
    width=1200
)

fig_facet.show()

## II/ Customer Behavior & Ride Characteristics

### Question 4: What are the most popular pickup and drop-off locations, and how do they change over time?


In [ ]:
query_popular_pickup_dropoff = """
SELECT *
FROM `smiling-rhythm-486213-b9.views_fordashboard.popular_pickup_dropoff`
"""
popular_pickup_dropoff_df = query_to_dataframe(query_popular_pickup_dropoff)
popular_pickup_dropoff_df.head()

In [ ]:
popular_pickup_dropoff_df.info()

In [ ]:
popular_pickup_dropoff_df.year.value_counts(normalize=True)

In [ ]:
filtered_popular_pickup_dropoff_df = popular_pickup_dropoff_df[(popular_pickup_dropoff_df['year'] >= 2020) & (popular_pickup_dropoff_df['year'] <= current_year)]

In [ ]:
monthly_trips = (
    filtered_popular_pickup_dropoff_df.groupby(["year", "month"])["total_trips"]
    .sum()
    .reset_index()
)
monthly_trips.head()

In [ ]:
monthly_trips["date"] = pd.to_datetime(
    monthly_trips["year"].astype(str) + "-" + monthly_trips["month"].astype(str)
)

fig = px.line(
    monthly_trips,
    x="date",
    y="total_trips",
    markers=True,
    title="Monthly Trends in Taxi Demand",
    labels={"total_trips": "Total Trips", "date": "Date"},
)

fig.update_traces(line=dict(width=3))
fig.update_layout(template="plotly_dark", xaxis_tickangle=-45)
fig.show()

In [ ]:
top_pickups = (
    filtered_popular_pickup_dropoff_df.groupby("pickup_zone")["total_trips"]
    .sum()
    .nlargest(10)
    .reset_index()
)

top_pickups.head()

In [ ]:
fig = px.bar(
    top_pickups,
    x="total_trips",
    y="pickup_zone",
    orientation="h",
    text="total_trips",
    title="Top 10 Pickup Locations",
    labels={"total_trips": "Total Trips", "pickup_zone": "Pickup Zone"},
)

fig.update_traces(marker_color="royalblue", texttemplate="%{text:,}", textposition="inside")
fig.update_layout(template="plotly_dark", xaxis_title="Total Trips")
fig.show()

In [ ]:
heatmap_data = (
    filtered_popular_pickup_dropoff_df.groupby(["pickup_borough", "weekday"])["total_trips"]
    .sum()
    .unstack()
)

heatmap_data.head()

In [ ]:
fig = go.Figure(
    data=go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns,
        y=heatmap_data.index,
        colorscale="viridis",
        colorbar=dict(title="Total Trips"),
    )
)

fig.update_layout(
    title="Taxi Demand Heatmap by Borough & Weekday",
    xaxis_title="Weekday",
    yaxis_title="Pickup Borough",
    template="plotly_dark",
)
fig.show()

### Question 5: What is the average trip distance, and how does it vary by borough, time of day, and season?


In [ ]:
query_avg_trip_distance_analysis = """
SELECT *
FROM `smiling-rhythm-486213-b9.views_fordashboard.avg_trip_distance_analysis`
"""
avg_trip_distance_analysis_df = query_to_dataframe(query_avg_trip_distance_analysis)
avg_trip_distance_analysis_df.head()

In [ ]:
filtered_avg_trip_distance_analysis_df = avg_trip_distance_analysis_df[(avg_trip_distance_analysis_df['year'] >= 2020) & (avg_trip_distance_analysis_df['year'] <= current_year)]
filtered_avg_trip_distance_analysis_df.info()

In [ ]:
filtered_avg_trip_distance_analysis_df["trip_date"] = pd.to_datetime(filtered_avg_trip_distance_analysis_df["trip_date"])

daily_avg_distance = (
    filtered_avg_trip_distance_analysis_df.groupby("trip_date")["avg_trip_distance"]
    .mean()
    .reset_index()
)

daily_avg_distance.head()

In [ ]:
fig = px.line(
    daily_avg_distance,
    x="trip_date",
    y="avg_trip_distance",
    title="Average Trip Distance Over Time",
    labels={"avg_trip_distance": "Avg Trip Distance (miles)", "trip_date": "Date"},
    markers=True,
)

fig.update_traces(line=dict(width=3))
fig.update_layout(template="plotly_dark", xaxis_tickangle=-45)
fig.show()

### Question 6: How many trips have only one passenger versus multiple passengers, and does this change seasonally?


In [ ]:
query_passenger_trends_by_season = """
SELECT *
FROM `smiling-rhythm-486213-b9.views_fordashboard.passenger_trends_by_season`
"""
passenger_trends_by_season_df = query_to_dataframe(query_passenger_trends_by_season)
passenger_trends_by_season_df.head()

In [ ]:
filtered_passenger_trends_by_season_df = passenger_trends_by_season_df[(passenger_trends_by_season_df['year'] >= 2020) & (passenger_trends_by_season_df['year'] <= current_year)]
filtered_passenger_trends_by_season_df.info()

In [ ]:
passenger_counts = filtered_passenger_trends_by_season_df.groupby("passenger_category")["total_trips"].sum().reset_index()
passenger_counts

In [ ]:
fig = px.pie(
    passenger_counts,
    names="passenger_category",
    values="total_trips",
    title="Proportion of Single vs. Multiple Passenger Trips",
    hole=0.4,
    color="passenger_category",
    color_discrete_map={"Single Passenger": "red", "Multiple Passengers": "blue"}
)

fig.update_traces(textinfo="percent+label")
fig.show()

In [ ]:
treemap_data = filtered_passenger_trends_by_season_df.groupby(["season", "passenger_category"])["total_trips"].sum().reset_index()

season_colors = {
    "Winter": "blue",
    "Spring": "green",
    "Summer": "yellow",
    "Fall": "orange",
}

fig = px.treemap(
    treemap_data,
    path=["season", "passenger_category"],
    values="total_trips",
    color="season",
    color_discrete_map=season_colors,
    title="Passenger Distribution by Season (Treemap)",
)

fig.update_traces(textinfo="label+percent entry")
fig.show()

In [ ]:
filtered_passenger_trends_by_season_df["trip_date"] = pd.to_datetime(filtered_passenger_trends_by_season_df["trip_date"])

In [ ]:
fig = px.area(
    filtered_passenger_trends_by_season_df,
    x="trip_date",
    y="total_trips",
    color="passenger_category",
    title="Proportion of Single vs. Multiple Passenger Trips Over Time",
    labels={"total_trips": "Total Trips", "trip_date": "Date", "passenger_category": "Passenger Category"},
)

fig.update_layout(template="plotly_dark")
fig.show()